# Word Embeddings

Dans cette démonstration, nous utilisons les word embeddings pré-entraînés de [spaCy](https://spacy.io/), puis nous apprenons les nôtres sur un corpus de forums, avec un skip-gram écrit en PyTorch.

In [1]:
!python -m spacy download fr_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 MB 20.2 MB/s  0:00:29
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')


In [2]:
import collections
import itertools
import pathlib
import re
import time

import numpy as np
import spacy
import torch

## Les vecteurs de spaCy

`fr_core_news_lg`, le grand modèle français de spaCy, associe 500 000 mots à autant de vecteurs de 300 dimensions. Chaque mot d'un texte analysé a son vecteur, et le texte entier a la moyenne des siens.

In [3]:
nlp = spacy.load("fr_core_news_lg")
doc = nlp("Ceci est une phrase que je traite avec spaCy")

print(doc[3].text, doc[3].vector.shape)
print(doc[3].vector[:10])

print("Vecteur moyen de la phrase :", doc.vector.shape)
print(doc.vector[:10])

phrase (300,)
[ 1.9004    0.97428  -0.014774  0.72923   2.3348    4.0516    0.3782
  0.70911  -2.0879    0.67758 ]
Vecteur moyen de la phrase : (300,)
[ 0.59239006 -0.35560772 -1.764606    0.5883033   2.4253945   3.519517
  0.8659423   1.2797602   0.1485911   0.1199744 ]


## Une table d'embeddings

La classe `Embeddings` range les vecteurs dans une table, un par ligne, ramenés à la longueur 1 : le produit scalaire de deux lignes est alors leur cosinus. Elle servira aussi aux vecteurs que nous apprendrons.

In [4]:
class Embeddings:
  """Une table d'embeddings : un vecteur de longueur 1 par mot."""

  def __init__(self, lignes, table):
    self.lignes = lignes  # mot -> numéro de sa ligne dans la table
    self.mots = {}  # numéro de ligne -> mot (en minuscules s'il y en a plusieurs)
    for mot, ligne in lignes.items():
      if ligne not in self.mots or mot.islower():
        self.mots[ligne] = mot
    # Lignes de longueur 1 : le produit scalaire de deux lignes est leur cosinus
    self.table = table / np.linalg.norm(table, axis=1, keepdims=True)

  def __contains__(self, mot):
    return mot in self.lignes

  def __getitem__(self, mot):
    return self.table[self.lignes[mot]]

  def voisins(self, vecteur, n=5):
    """Les n mots les plus proches de `vecteur` : des couples (mot, cosinus)."""
    cosinus = self.table @ (vecteur / np.linalg.norm(vecteur))
    return [
      (self.mots[i], round(float(cosinus[i]), 2)) for i in np.argsort(-cosinus)[:n]
    ]

In [5]:
vecteurs = nlp.vocab.vectors
modele = Embeddings(
  {nlp.vocab.strings[cle]: ligne for cle, ligne in vecteurs.key2row.items()},
  vecteurs.data,
)
print(len(modele.lignes), "mots,", modele.table.shape)

500000 mots, (500000, 300)


### Mots présents et absents

La casse compte : `Berlin` et `berlin` ont chacun leur vecteur, et un mot mal orthographié n'en a aucun. Un texte se prétraite comme ceux qui ont servi à apprendre les vecteurs.

In [6]:
for mot in ["Berlin", "berlin", "Windoows"]:
  print(mot, mot in modele)

Berlin True
berlin True
Windoows False


### Voisins

Les voisins d'un mot sont les lignes de plus grand cosinus avec la sienne. `dos` et `DOS` sont deux mots différents.

In [7]:
for mot in ["Berlin", "berlin", "dos", "DOS"]:
  print(mot, modele.voisins(modele[mot]))

Berlin [('Berlin', 1.0), ('Munich', 0.79), ('Hambourg', 0.77), ('Berlinois', 0.76), ('Dresde', 0.75)]
berlin [('berlin', 1.0), ('hagen', 0.61), ('frankfurt', 0.61), ('munich', 0.59), ('francfort', 0.59)]
dos [('dos', 1.0), ('cou', 0.69), ('épaule', 0.69), ('poignet', 0.67), ('ventre', 0.67)]
DOS [('DOS', 1.0), ('UNIX', 0.68), ('Unix', 0.68), ('MS', 0.64), ('OS/2', 0.64)]


### Similarité de deux mots

Le produit scalaire de deux vecteurs de longueur 1 est leur cosinus.

In [8]:
print(modele["Berlin"] @ modele["Munich"])
print(modele["Berlin"] @ modele["Paris"])

0.7942655
0.5392447


## Analogies

`x` est à `y` ce que `z` est à ? On cherche les voisins de `y - x + z`. *Homme* est à *femme* ce que *roi* est à :

In [9]:
modele.voisins(modele["femme"] - modele["homme"] + modele["roi"])

[('roi', 0.75),
 ('reine', 0.68),
 ('princesse', 0.6),
 ('régente', 0.56),
 ('Roi', 0.56)]

En tête vient `roi`, un mot de la question : `femme - homme` déplace peu le vecteur de `roi`. La fonction `analogie` écarte donc les mots de la question.

In [10]:
def analogie(modele, x, y, z):
  # x est à y ce que z est à ? Les mots de la question sont écartés
  vecteur = modele[y] - modele[x] + modele[z]
  # Les lignes des mots de la question
  question = {modele.lignes[mot] for mot in (x, y, z)}
  proches = modele.voisins(vecteur, n=8)
  return [(mot, c) for mot, c in proches if modele.lignes[mot] not in question][:5]


analogie(modele, "homme", "femme", "roi")

[('reine', 0.68),
 ('princesse', 0.6),
 ('régente', 0.56),
 ('Roi', 0.56),
 ('Brunehilde', 0.54)]

### Une réussite, un échec, un biais

*France* est à *Paris* ce que *Japon* est à :

In [11]:
analogie(modele, "France", "Paris", "Japon")

[('Tokyo', 0.7),
 ('Osaka', 0.62),
 ('Yokohama', 0.59),
 ('Tokyoïte', 0.59),
 ('Tôkyô', 0.57)]

Avec l'Italie, Rome n'est pas parmi les cinq premiers : une analogie n'est qu'un voisinage, et d'autres villes italiennes sont plus proches.

In [12]:
analogie(modele, "France", "Paris", "Italie")

[('Bologne', 0.67),
 ('Venise', 0.66),
 ('Milan', 0.66),
 ('Naples', 0.65),
 ('Turin', 0.65)]

Les vecteurs reprennent les associations des textes dont ils sont tirés, stéréotypes compris. Appris sur le web, ils comptent aussi des graphies rares (`mèdecin`, `linfirmière`).

In [13]:
analogie(modele, "homme", "femme", "médecin")

[('mèdecin', 0.68),
 ('pédiatre', 0.66),
 ('gynécologue', 0.65),
 ('linfirmière', 0.65),
 ('ex-médecin', 0.65)]

## Apprendre ses propres embeddings

### Le corpus 20 Newsgroups

Environ 18 800 messages de forums en anglais, répartis en 20 thèmes.

In [14]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

--2026-11-05 18:37:05--  https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz [following]
--2026-11-05 18:37:06--  https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14464277 (14M) [application/octet-stream]
Saving to: ‘20news-bydate.tar.gz’

20news-bydate.tar.g 100%[===================>]  13.79M  4.71MB/s    in 2.9s    

2026-11-05 18:37:11 (4.71

In [15]:
!ls 20news-bydate-test/

alt.atheism		  rec.autos	      sci.space
comp.graphics		  rec.motorcycles     soc.religion.christian
comp.os.ms-windows.misc   rec.sport.baseball  talk.politics.guns
comp.sys.ibm.pc.hardware  rec.sport.hockey    talk.politics.mideast
comp.sys.mac.hardware	  sci.crypt	      talk.politics.misc
comp.windows.x		  sci.electronics     talk.religion.misc
misc.forsale		  sci.med


In [16]:
contenus = [
  p.read_text(encoding="latin-1") for p in pathlib.Path(".").glob("20news-bydate-*/*/*")
]
print(f"{len(contenus)} textes récupérés")

18846 textes récupérés


In [17]:
print(contenus[1])

From: jagst18+@pitt.edu (Josh A Grossman)
Subject: Re: Dillon puts foot in mouth, film at 11
Organization: The Zets at the University of Pittsburgh (aka The Pitt)
Lines: 17


I have also heard about HCI claiming thant anyone they  get an address
from is a member.  If this is the case their membership rolls are
grossly inflated and we should not call them and give them a name
and address to add to their already false rolls.  Perhaps
if you could get a copy of their existing membership, then pretend to
be an existing member, do that several thousand times, you could
hurt HCI.  But names are power.  Remeber the NRA uses the fact
that it has 3 million paid members in order to flex its muscles.

Perhaps politicians don't realize the lying tactics of HCI, wait a
minute, HCI learned it from politicians....

Later,
Josh





`pretraiter` retire l'en-tête d'un message et les lignes qui annoncent une citation (« … writes: »), puis découpe le reste en mots, sur les espaces et la ponctuation. La casse reste.

In [18]:
def pretraiter(contenu: str) -> list[str]:
  debut = contenu.find("\n\n")
  if debut > 0:
    contenu = contenu[debut + 2 :]
  mots_par_ligne = (
    re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", ligne).split()
    for ligne in contenu.splitlines()
    if not ligne.endswith("writes:")
  )
  return list(itertools.chain.from_iterable(mots_par_ligne))


textes = [pretraiter(contenu) for contenu in contenus]
print(textes[0])

['Do', 'you', 'know', 'how', 'many', 'deaths', 'each', 'year', 'are', 'caused', 'by', 'self', 'inflicted', 'gun', 'shot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'There', 'are', 'roughly', '1200', 'fatal', 'firearms', 'related', 'accidents', 'each', 'year', 'The', 'large', 'majority', 'involve', 'rifles', 'and', 'shotgun', 'there', 'are', 'under', '500', 'fatal', 'handgun', 'accidents', 'each', 'year', 'I', 'really', 'doubt', 'all', 'of', 'those', 'occur', 'while', 'the', 'pistol', 'is', 'holstered', 'so', 'the', 'number', 'of', 'self', 'inflicted', 'gunshot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'is', 'probably', 'well', 'under', '250', 'per', 'year', "I'm", 'neither', 'a', 'doctor', 'nor', 'a', 'firearms', 'tech', 'expert', 'but', 'it', 'would', 'seem', 'that', 'given', 'the', 'way', 'a', 'holstered', 'gun', 'points', 'accidental', 'injuries', 'inflicted', 'that', 'way', 'would', 'be', 'among', 'the', 'least', 'lethal', 'David', 'Veal', 'Univ', 

### Les expressions fréquentes

*New York* désigne une seule chose : mieux vaut un mot `New_York` que deux. Deux mots qui se suivent bien plus souvent que leurs fréquences ne le laissent attendre forment une expression, d'après le score de [Mikolov et al. (2013)](https://arxiv.org/abs/1310.4546) : $(\text{compte}(ab) - \delta) / (\text{compte}(a) \times \text{compte}(b))$. Quelques secondes suffisent.

In [19]:
# Deux mots qui se suivent bien plus souvent que le hasard ne le voudrait forment
# une expression : score de Mikolov et al. (2013), (compte(ab) - δ) / (compte(a) × compte(b))
comptes_mots = collections.Counter(mot for texte in textes for mot in texte)
comptes_paires = collections.Counter(
  p for texte in textes for p in zip(texte, texte[1:])
)
total = sum(comptes_mots.values())  # × total : le score devient un rapport au hasard
expressions = {
  (a, b)
  for (a, b), compte in comptes_paires.items()
  if (compte - 5) * total / (comptes_mots[a] * comptes_mots[b]) > 100
}


def grouper(texte):
  # Remplace chaque paire d'une expression par un seul mot, a_b
  groupe = []
  for mot in texte:
    if groupe and (groupe[-1], mot) in expressions:
      groupe[-1] = groupe[-1] + "_" + mot
    else:
      groupe.append(mot)
  return groupe


textes_groupes = [grouper(texte) for texte in textes]

In [20]:
print(len(expressions), "expressions")
print(sorted(a + "_" + b for a, b in expressions if a == "New"))
print(textes_groupes[0][:20])

20429 expressions
['New_Age', 'New_Brunswick', 'New_England', 'New_Hampshire', 'New_Haven', 'New_Hobgoblin', 'New_Jersey', 'New_Mexico', 'New_Mutants', 'New_Orleans', 'New_Regime', 'New_Testament', 'New_Warriors', 'New_York', 'New_Yorker', 'New_Zealand']
['Do', 'you', 'know', 'how', 'many', 'deaths', 'each', 'year', 'are', 'caused_by', 'self', 'inflicted', 'gun', 'shot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters']


### Un skip-gram en PyTorch

On entraînait ces embeddings, dits statiques, avec une bibliothèque dédiée, comme gensim ou fastText : c'est la manière de faire historique. Aujourd'hui, les représentations s'apprennent à l'intérieur des modèles (les transformers), et [gensim](https://github.com/piskvorky/gensim) est en maintenance, sans nouvelle fonctionnalité. Un skip-gram avec échantillonnage négatif tient en quelques lignes de PyTorch.

Le vocabulaire garde les mots vus au moins 5 fois, et le corpus devient une seule suite de numéros de mots. Un mot très fréquent est le plus souvent retiré : il n'apprend rien de plus à ses voisins.

In [21]:
DIMENSION, FENETRE, NEGATIFS, LOT = 100, 5, 5, 4096
EPOQUES = 3

# Vocabulaire : les mots vus au moins 5 fois, du plus fréquent au plus rare
comptes = collections.Counter(mot for texte in textes_groupes for mot in texte)
vocabulaire = [mot for mot, compte in comptes.most_common() if compte >= 5]
numeros = {mot: numero for numero, mot in enumerate(vocabulaire)}
frequences = torch.tensor([comptes[mot] for mot in vocabulaire], dtype=torch.float)
# Le corpus : une seule suite de numéros de mots
corpus = torch.tensor(
  [numeros[mot] for texte in textes_groupes for mot in texte if mot in numeros]
)
# Sous-échantillonnage : un mot très fréquent (the, of…) est retiré le plus souvent
torch.manual_seed(0)
garde = torch.rand(len(corpus)) < torch.sqrt(
  1e-3 * frequences.sum() / frequences[corpus]
)
corpus = corpus[garde]
print(len(vocabulaire), "mots,", len(corpus), "occurrences gardées")

57189 mots, 3501077 occurrences gardées


Le modèle a deux tables : le vecteur d'un mot au centre de la fenêtre, et son vecteur quand il en est un voisin. Le score d'un couple est le produit scalaire de leurs deux vecteurs.

In [22]:
class SkipGram(torch.nn.Module):
  def __init__(self, taille_vocabulaire, dimension):
    super().__init__()
    # Deux tables : le vecteur d'un mot au centre, et son vecteur comme voisin
    self.centres = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    self.contextes = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    torch.nn.init.uniform_(self.centres.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contextes.weight)

  def forward(self, centre, contexte):
    # Score d'un couple : le produit scalaire de ses deux vecteurs
    return (self.centres(centre) * self.contextes(contexte)).sum(dim=-1)

La perte pousse le score d'un vrai voisin vers le haut, et celui de mots tirés au hasard, les négatifs, vers le bas.

In [23]:
def perte(skipgram, centre, contexte, negatifs):
  # -log σ(score du vrai voisin) - Σ log σ(-score d'un négatif), en moyenne sur le lot
  vrai = torch.nn.functional.logsigmoid(skipgram(centre, contexte))
  faux = torch.nn.functional.logsigmoid(-skipgram(centre[:, None], negatifs)).sum(dim=1)
  return -(vrai + faux).mean()

L'entraînement : chaque lot tire des positions du corpus, un voisin dans la fenêtre de chacune, et des négatifs. Une époque prend environ une minute sur deux processeurs, comme ceux de Colab.

In [24]:
skipgram = SkipGram(len(vocabulaire), DIMENSION)
optimiseur = torch.optim.SparseAdam(skipgram.parameters(), lr=0.003)
tirage_negatifs = frequences**0.75  # un peu plus de chances aux mots rares

debut = time.time()
for epoque in range(EPOQUES):
  total = 0.0
  lots = torch.randperm(len(corpus) - 2 * FENETRE).add(FENETRE).split(LOT)
  for positions in lots:
    # Un voisin tiré dans la fenêtre : à 1..FENETRE mots, à gauche ou à droite
    decalage = torch.randint(1, FENETRE + 1, positions.shape)
    decalage = decalage * (2 * torch.randint(0, 2, positions.shape) - 1)
    centre, contexte = corpus[positions], corpus[positions + decalage]
    negatifs = torch.multinomial(
      tirage_negatifs, len(positions) * NEGATIFS, replacement=True
    )
    valeur = perte(skipgram, centre, contexte, negatifs.view(-1, NEGATIFS))
    optimiseur.zero_grad()
    valeur.backward()
    optimiseur.step()
    total += valeur.item()
  print(
    f"époque {epoque + 1} : perte {total / len(lots):.3f}, {time.time() - debut:.0f} s"
  )

époque 1 : perte 2.820, 70 s
époque 2 : perte 2.445, 143 s
époque 3 : perte 2.368, 213 s


### Les vecteurs appris

La table des centres prend place dans la même classe que les vecteurs de spaCy. Sur ce corpus du domaine, les voisins de `Windows` et `DOS` viennent des forums informatiques ; `New_York`, un mot de notre vocabulaire, manque à celui de spaCy, dont le tokeniseur coupe *New York* en deux.

In [25]:
modele_forums = Embeddings(numeros, skipgram.centres.weight.detach().numpy())

for mot in ["Windows", "DOS", "New_York"]:
  print(mot, modele_forums.voisins(modele_forums[mot]))
  if mot in modele:
    print("  spaCy :", modele.voisins(modele[mot]))
  else:
    print("  spaCy : absent")

Windows [('Windows', 1.0), ('Windows_NT', 0.87), ('Excel', 0.87), ('OS_2', 0.86), ('windows', 0.86)]
  spaCy : [('Windows', 1.0), ('Windows8', 0.96), ('Windows7', 0.96), ('Windows10', 0.95), ('WindowsPhone', 0.9)]
DOS [('DOS', 1.0), ('MS_DOS', 0.84), ('MS_Windows', 0.84), ('tcp_ip', 0.83), ('Windows_NT', 0.82)]
  spaCy : [('DOS', 1.0), ('UNIX', 0.68), ('Unix', 0.68), ('MS', 0.64), ('OS/2', 0.64)]
New_York [('New_York', 1.0), ('replaced_Jimmy', 0.86), ('League', 0.85), ('1982', 0.85), ('Carter', 0.84)]
  spaCy : absent


Cinq millions de mots ne suffisent pas aux analogies :

In [26]:
analogie(modele_forums, "man", "woman", "king")

[('Women', 0.87),
 ('west', 0.86),
 ('rose', 0.84),
 ('voted', 0.84),
 ('elite', 0.83)]